# Modulo 8 — Sesion 6 · Ejercicio B

| | |
|---|---|
| **Tipo** | Ejercicio B — Independiente |
| **Tiempo** | 50 minutos |
| **Modalidad** | Cada participante prepara y ensaya de forma individual |
| **Producto esperado** | Guion completado + ensayo cronometrado + documento Word de conclusiones |

### Objetivo
Cada participante prepara su presentacion de 5 minutos con el dashboard
publicado, llenando el guion con sus numeros reales,
construyendo la pestana de conclusiones e integrando el documento Word.

- Dar 8 min para completar el guion en silencio
- Dar 15 min para generar el documento Word
- Dar 15 min para primer ensayo en voz alta (cronometrado)
- Dar 12 min para que 2-3 voluntarios presenten frente al grupo
- Evaluar con la rubrica: ¿un externo entiende sin contexto previo?

### Cierre del modulo
Despues de las presentaciones, lanzar la pregunta de cierre:
*"¿Cual es la diferencia entre un dashboard que muestra datos
y un dashboard que cuenta una historia?"*


## ✏️ TU TURNO — Presentacion de 5 minutos

**Pasos:**
1. Ejecuta la celda de numeros — obtén tus datos reales
2. Lee el guion y personaliza los valores entre corchetes
3. Genera el documento Word de conclusiones
4. Ensaya el guion en voz alta — cronometrate
5. Ajusta lo que no quede claro al escucharlo

In [6]:
# PASO 1 — Calcular los numeros reales del dashboard para llenar el guion
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('../hr_ibm.csv')

total = len(df)
rotaron = (df['Attrition'] == 'Yes').sum()
tasa_global = rotaron / total * 100
sal_promedio = df['MonthlyIncome'].mean()
costo_renuncia = sal_promedio * 1.5 * 12

rot_depto = df.groupby('Department')['Attrition'].apply(
    lambda x : (x=='Yes').mean()*100
)

depto_critico = rot_depto.idxmax()
tasa_critica = rot_depto.max()

tasa_prom_dep = rot_depto.mean()

ratio_critico = tasa_critica / tasa_prom_dep

rot_ot = df.groupby('OverTime')['Attrition'].apply(
    lambda x : (x=='Yes').mean()*100
)

ratio_ot = rot_ot.get('Yes', 0) / max(rot_ot.get('No', 1), 0.01)


def calcular_score(df):
    d = df.copy()
    d['n_overtime'] = (d['OverTime'] == 'Yes').astype(int)*100
    d['n_jobsat'] = (4 - d['JobSatisfaction']) / 3 * 100
    d['n_years'] = ((3 - d['YearsAtCompany'])/3 * 100).clip(0,100)
    d['n_wlb'] = (4 - d['WorkLifeBalance']) / 3 * 100
    d['n_envsat'] = (4 - d['EnvironmentSatisfaction'])/3 * 100

    d['score_riesgo'] = (
        d['n_overtime']*0.30 + d['n_jobsat']*0.25 + d['n_years']*0.20 + d['n_wlb']*0.15 + d['n_envsat']*0.10
     ).round(1)

    d['nivel_riesgo'] = pd.cut(d['score_riesgo'], bins=[0, 39,69,100], labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)
    return d

df_s = calcular_score(df)
top3 = df_s.nlargest(3, 'score_riesgo')[['JobRole', 'Department', 'score_riesgo']]
n_alto = (df_s['score_riesgo'] > 70).sum()

print("Para el guión que se va a presentar o a entregar")
print(f"Total: {total:,} colaboradores")
print(f"Rotaron: {rotaron} personas ({tasa_global:.1f}%)")
print(f"Depto Critico : {depto_critico} ({tasa_critica:.1f}%)")
print(f"VS Promedio de Deptos: {tasa_prom_dep:.1f}% ({ratio_critico:.1f} x mayor)")
print(f"Razon de Tiempo Extra Si: {ratio_ot:.1f} x mas rotación")
print(f"Salario Promedio: ${sal_promedio:,.0f}/mes")
print(f"Empleados riesgo alto : {n_alto}")
print()
print("Top de 3 empleados en riesgo")
print(top3.to_string(index=False))


print("Grafica de Rotación por Departamento")

df_dep = rot_depto.reset_index()
df_dep.columns = ['Department', 'Tasa_rotacion']
df_dep = df_dep.sort_values('Tasa_rotacion', ascending=False)

# Anotacion 1: linea de referencia del sector
fig_con = px.bar(df_dep, x='Department', y='Tasa_rotacion',
                 title='Tasa de Rotación por Departamento (%)', 
                 color='Tasa_rotacion', text_auto='.1f', color_continuous_scale='RdYlGn_r',
                 labels={'Tasa_rotacion': 'Tasa (%)' , 'Department': 'Departamento'})

fig_con.add_hline(y=10, line_dash='dot', line_color="#70C1E9", line_width=2,
                  annotation_text='Meta sector: 10%',
                  annotation_position='top right',
                  annotation_font_color='#F5F717', annotation_font_size=12)


# Anotacion 2: etiqueta sobre la barra critica
fig_con.add_annotation(
    x=depto_critico, y=tasa_critica,
    text=f'{depto_critico}: {tasa_critica:.1f}% -- {tasa_critica-10:-1f} pts sobre la meta',
    showarrow=True, arrowhead=2, arrowcolor='#8B0000', bgcolor='#FFBEEE', 
    font=dict(color='#8B0000', size=12), yshift=10
)


fig_con.update_layout(height=420, showlegend=False)
fig_con.show()


print("Grafica 2 Impacto de las Horas Extra en la Rotación")

df_ot = rot_ot.reset_index()
df_ot.columns = ['OverTime', 'Tasa_rotacion']

fig_ot = px.bar(
    df_ot, x='OverTime', y='Tasa_rotacion',
    title=f'Rotacion segun hrs extra ({ratio_ot:.1f}x)',
    labels={'Tasa_rotacion':'Tasa (%)', 'OverTime': 'Horas extra'},
    color='OverTime',
    color_discrete_map={'Yes': '#C62828', 'No': '#2E7D32'},
    text_auto='.1f'
)
fig_ot.update_layout(height=350, showlegend=False)
fig_ot.show()


Para el guión que se va a presentar o a entregar
Total: 1,470 colaboradores
Rotaron: 237 personas (16.1%)
Depto Critico : Sales (20.6%)
VS Promedio de Deptos: 17.8% (1.2 x mayor)
Razon de Tiempo Extra Si: 2.9 x mas rotación
Salario Promedio: $6,503/mes
Empleados riesgo alto : 31

Top de 3 empleados en riesgo
              JobRole             Department  score_riesgo
      Sales Executive                  Sales          95.0
Laboratory Technician Research & Development          91.7
Laboratory Technician Research & Development          83.3
Grafica de Rotación por Departamento


Grafica 2 Impacto de las Horas Extra en la Rotación


In [ ]:
# PASO 2 — GUION DE 5 MINUTOS (completa con tus numeros)
guion = f"""
GUION DE PRESENTACION — DASHBOARD TECHNOVA
Tiempo total: 5 minutos | Cronometrar en el ensayo

--- MIN 1: CONTEXTO (30 segundos) ---
"El dataset de TechNova tiene {total:,} colaboradores.
La tasa de rotacion actual es del {tasa_global:.1f}%.
Eso representa {rotaron} personas reales que dejaron la empresa."

--- MIN 2: HALLAZGO 1 (90 segundos) ---
"El departamento con mayor rotacion es {depto_critico} con una tasa del {tasa_critica:.1f}%.
Eso es {ratio_critico:.1f} veces mayor que el promedio del resto de departamentos.
La linea de referencia muestra que estamos {tasa_critica-10:.1f} puntos
por encima del estandar del sector del 10%."

--- MIN 3: HALLAZGO 2 (90 segundos) ---
"Los empleados con horas extra tienen {ratio_ot:.1f} veces mayor probabilidad de renunciar.
En el scatter puede ver que los puntos rojos se concentran
en la zona de alta carga laboral."

--- MIN 4: RIESGO (30 segundos) ---
"En la pestana de Riesgo estan los 10 empleados con mayor probabilidad de renunciar.
Los 3 primeros son:
  1. {top3.iloc[0]['JobRole']} — {top3.iloc[0]['Department']}
  2. {top3.iloc[1]['JobRole']} — {top3.iloc[1]['Department']}
  3. {top3.iloc[2]['JobRole']} — {top3.iloc[2]['Department']}"

--- MIN 5: RECOMENDACION (30 segundos) ---
"Mi recomendacion: revision de condiciones de horas extra en {depto_critico}.
Una conversacion de retencion con los 3 primeros del ranking
podria evitar ${costo_renuncia*3:,.0f} en costos de contratacion."
"""

print(guion)
print("INSTRUCCION: Leer en voz alta y cronometrar. Ajustar si pasa de 5:30 minutos.")


In [ ]:
# PASO 3 — DOCUMENTO WORD DE CONCLUSIONES


# Encabezado



In [ ]:
# PASO 4 — REFLEXION FINAL DEL MODULO
print("Preguntas de reflexion para responder en las celdas Markdown siguientes:")
print()
print("1. ¿Que diferencia hay entre un dashboard que 'muestra datos'")
print("   y uno que 'cuenta una historia'? Da un ejemplo de tu propio dashboard.")
print()
print("2. ¿En que momento de la presentacion crees que el Director tomaria")
print("   la decision de actuar? ¿Por que en ese momento y no antes?")
print()
print("3. ¿Cambiaria tu recomendacion si filtraras solo el departamento critico?")
print("   ¿Que nueva informacion aportaria ese filtro?")


### Responde aqui:

**Pregunta 1:** ¿Que diferencia hay entre mostrar datos y contar una historia?

*Tu respuesta:*

---

**Pregunta 2:** ¿En que momento la presentacion convenceria al Director de actuar?

*Tu respuesta:*

---

**Pregunta 3:** ¿Cambiaria tu recomendacion si filtraras solo el departamento critico?

*Tu respuesta:*

### Autoevaluacion
| Criterio | Cumple |
|----------|--------|
| El guion tiene los numeros reales del dashboard (no ejemplos genericos) | ☐ |
| El guion fue ensayado en voz alta y cronometrado | ☐ |
| El documento Word tiene los 4 actos narrativos | ☐ |
| El documento Word usa lenguaje ejecutivo (sin terminos tecnicos) | ☐ |
| La recomendacion incluye el costo estimado de no actuar | ☐ |
| Las 3 preguntas de reflexion fueron respondidas | ☐ |